# Workflow: Visium CRC {#sec-seq-workflow-visium-crc}



## Preamble

### Introduction

In this demo, we will be analyzing Visium data on a human colorectal cancer biopsy from @deOliveira2025-high-def. Rather than recapitulating all possible analyses, our goal is to highlight those that might be of particular interest in the context of these data.

### Dependencies

In [ ]:
library(AUCell)
library(BiocParallel)
library(DropletUtils)
library(ggspavis)
library(igraph)
library(jsonlite)
library(msigdbr)
library(OSTA.data)
library(patchwork)
library(pheatmap)
library(scater)
library(scrapper)
library(spacexr)
library(SpatialExperiment)
library(VisiumIO)
# specify whether/how to 
# perform parallelization
bp <- MulticoreParam(th <- 4)
# set seed for random number generation
# in order to make results reproducible
set.seed(194849)

## Data import {#sec-seq-workflow-visium-crc-load-data}

In [ ]:
# retrieve dataset from OSF repo
id <- "Visium_HumanColon_Oliveira"
pa <- OSTA.data_load(id)
dir.create(td <- tempfile())
unzip(pa, exdir=td)

# read into 'SpatialExperiment'
obj <- TENxVisium(
    spacerangerOut=file.path(td, "outs"), 
    format="h5", 
    images="lowres")
(spe <- import(obj))

In [ ]:
xy <- spatialCoords(spe)*scaleFactors(spe)
xs <- range(xy[, 1]); ys <- nrow(imgRaster(spe))-range(xy[, 2])
box <- geom_rect(
    xmin=xs[1], xmax=xs[2], ymin=ys[1], ymax=ys[2],
    col="black", fill=NA, linetype=2, linewidth=2/3)
plotVisium(spe, spots=FALSE, point_size=.75) + box +
plotVisium(spe, point_size=1, zoom=TRUE) +
plot_layout(nrow=1) & facet_null()

## Quality control

In [ ]:
# use gene symbols as feature names
rownames(spe) <- make.unique(rowData(spe)$Symbol)
# add per-cell quality control metrics & determine 
# outliers via thresholding on MAD from the median
sub <- list(mt=grep("^MT-", rownames(spe)))
spe <- quickRnaQc.se(spe, subsets=sub)
spe$discard <- !spe$keep
spe$log_sum <- log1p(spe$sum)
spe$mt_prop <- spe$subset.proportion.mt

In [ ]:
plotCoords(spe, annotate="log_sum") + ggtitle("log-library size") +
plotCoords(spe, annotate="mt_prop") + ggtitle("% mitochondrial") +
plot_layout(nrow=1) & theme(
    legend.key.width=unit(0.5, "lines"),
    legend.key.height=unit(1, "lines")) &
    scale_color_gradientn(colors=pals::jet())

In [ ]:
# tabulate # & % of cells that'd be 
# discarded for different reasons
ths <- metadata(spe)$qc$thresholds
ols <- data.frame(
    low_sum=spe$sum < ths$sum,
    low_detected=spe$detected < ths$detected,
    high_mt_prop=spe$mt_prop > ths$subset.proportion,
    discard=spe$discard)
data.frame(
    check.names=FALSE,
    `#`=apply(ols, 2, sum), 
    `%`=round(100*apply(ols, 2, mean), 2))

Let's see which spots would be excluded according to the above criteria:

In [ ]:
#| code-fold: true
colData(spe)[names(ols)] <- ols
lapply(names(ols), \(.) 
    plotCoords(spe, annotate=.) + ggtitle(.)) |>
    wrap_plots(nrow=1, guides="collect") &
    guides(col=guide_legend(override.aes=list(size=3))) &
    scale_color_manual("discard", values=c("lavender", "purple")) &
    theme(plot.title=element_text(hjust=0.5), legend.key.size=unit(0, "lines"))

It seems like low-quality spots are highly spatially organized, so that might might encourage us to not remove them, for now. We will see further below how the quality control metrics used here, and spots deemed to be `discarded`, are distributed across (transcription-based) clusters.

## Processing

In [ ]:
# log-library size normalization
spe <- normalizeRnaCounts.se(spe)
# highly variable feature selection
spe <- chooseRnaHvgs.se(spe, top=2e3,
    more.var.args=list(use.min.width=TRUE))
# principal component analysis
spe <- runPca.se(spe, features=rowData(spe)$hvg)

## Clustering

As an unsupervised approach, we perform shared nearest-neighbor (SNN) graph-based clustering using the Leiden community detection algorithm.

In [ ]:
# PCA-based shared nearest-neighbor (SNN) graph;
# cluster via Leiden community detection algorithm
spe <- clusterGraph.se(spe,
    output.name="Leiden",
    method="leiden", resolution=0.5,
    more.build.args=list(weight.scheme="jaccard"))
table(spe$Leiden)

## Deconvolution

In a complementary approach, we deconvolute spot measurements using (annotated) reference single-cell data provided by the authors. Let's first retrieve these data, alongside corresponding cell metadata, which includes low- (`Level1`) and high-resolution (`Level2`) annotations into 10 and 32 subpopulations, respectively.

In [ ]:
# retrieve dataset from OSF repo
id <- "Chromium_HumanColon_Oliveira"
pa <- OSTA.data_load(id)
dir.create(td <- tempfile())
unzip(pa, exdir=td)

In [ ]:
# read into 'SingleCellExperiment'
fs <- list.files(td, full.names=TRUE)
h5 <- grep("h5$", fs, value=TRUE)
sce <- read10xCounts(h5, col.names=TRUE)
# add cell metadata
csv <- grep("csv$", fs, value=TRUE)
cd <- read.csv(csv, row.names=1)
colData(sce)[names(cd)] <- cd[colnames(sce), ]
# use gene symbols as feature names
rownames(sce) <- make.unique(rowData(sce)$Symbol)
# exclude cells deemed to be of low-quality
sce <- sce[, sce$QCFilter == "Keep"]
# tabulate subpopulations
table(sce$Level1)

[Here, we filter the reference data to contain only cells from the same patient, and downsample to retain a limited number of cells per subpopulation. This is not strictly necessary, assuming that clusters are transcriptionally stable across patients, but helps with reducing runtime and memory consumption here.]{.aside} Next, we perform deconvolution with `r BiocStyle::Biocpkg("spacexr")`'s (RCTD) [@Cable2022-RCTD].
By default, `runRctd()`'s `rctd_mode="doublet"`, i.e., at most two subpopulations are fit per pixel; here, we set `rctd_mode="full"` in order to allow for an arbitrary number of subpopulations to be fit instead.

In [ ]:
# prep reference data (Chromium);
# subset cells from same patient
.sce <- sce[, grepl("P2", sce$Patient)]
# downsample to at most 2,000 cells per cluster
cs <- split(seq_len(ncol(.sce)), .sce$Level1)
cs <- lapply(cs, \(.) sample(., min(length(.), 2e3)))
.sce <- .sce[, unlist(cs)]
# run 'RCTD' deconvolution
rctd_data <- createRctd(spe, .sce, cell_type_col="Level1")
(res <- runRctd(rctd_data, max_cores=th, rctd_mode="full"))

Weights inferred by `RCTD` should be normalized such that proportions of cell types sum to 1 in each spot:

In [ ]:
# scale weights such that they sum to 1
ws <- assay(res)
ws <- sweep(ws, 2, colSums(ws), `/`)
# add proportion estimates as metadata
ws <- data.frame(t(as.matrix(ws)))
colData(spe)[names(ws)] <- ws[colnames(spe), ]

For comparison with unsupervised clustering (SNN-based Leiden), we also include assignments we would obtain if we were to assign spots the most frequent label (in terms of deconvolution estimates):

In [ ]:
ids <- names(ws)[apply(ws, 1, which.max)]
table(spe$RCTD <- factor(ids), spe$Leiden)

We can also compartmentalize the tissue into broad biological compartments; here, by grouping `RCTD`-based subpopulation assignments into four classes:

In [ ]:
lab <- list(
    tum="Tumor",
    epi="Intestinal.Epithelial",
    imm=c("B.cells", "T.cells", "Myeloid"),
    str=c("Endothelial", "Fibroblast", "Smooth.Muscle"))
idx <- match(spe$RCTD, unlist(lab))
lab <- rep.int(names(lab), sapply(lab, length))
table(spe$Domain <- factor(lab[idx]))

## Exploratory

Let's visualize deconvolution weights in space, i.e., coloring by the proportion of a given cell type estimated to fall within a given spot:

In [ ]:
#| code-fold: true
lapply(names(ws), \(.) 
    plotCoords(spe, annotate=.)) |>
    wrap_plots(nrow=3) & theme(
    legend.key.width=unit(0.5, "lines"),
    legend.key.height=unit(1, "lines")) &
    scale_color_gradientn(colors=pals::jet())

In [ ]:
#| code-fold: true
lapply(c("Leiden", "Domain", "RCTD"), 
    \(.) plotCoords(spe, annotate=.)) |>
    wrap_plots(nrow=1) &
    theme(legend.key.size=unit(0, "lines")) &
    scale_color_manual(values=unname(pals::trubetskoy()))

To help characterize subpopulations from unsupervised clustering, we can view their distribution across deconvolution-based clusters and broad domains; e.g., tumor spots are quite diverse, while smooth muscle spots and (normal) epithelia map almost completely to a single cluster:

In [ ]:
#| code-fold: true
cd <- data.frame(colData(spe))
df <- as.data.frame(with(cd, table(RCTD, Leiden)))
fd <- as.data.frame(with(cd, table(Domain, Leiden)))
ggplot(df, aes(Freq, RCTD, fill=Leiden)) + ggtitle("RCTD") +
ggplot(fd, aes(Freq, Domain, fill=Leiden)) + ggtitle("Domain") +
plot_layout(nrow=1, guides="collect") &
    labs(x="Proportion", y=NULL) &
    coord_cartesian(expand=FALSE) &
    geom_col(width=1, col="white", position="fill") &
    scale_fill_manual(values=unname(pals::trubetskoy())) &
    theme_minimal() & theme(aspect.ratio=1,
        legend.key.size=unit(2/3, "lines"),
        plot.title=element_text(hjust=0.5))

Let's inspect the key drivers of (expression) variability in terms of PCs. Considering clustering and deconvolution results from above, we can see that

-   PC1 distinguishes stromal from both normal and malignant epithelia;
-   PC2 clearly separates (normal) intestinal epithelium from all else;
-   PC3 captures a fibroblast-rich region, and normal epithelia;
-   PC5 separates fibroblasts and smooth muscle cells; etc.

In [ ]:
#| code-fold: true
# add PCs as cell metadata
pcs <- reducedDim(spe, "PCA")
colnames(pcs) <- paste0("PC", seq(ncol(pcs)))
colData(spe)[colnames(pcs)] <- pcs

# visualize PCs 1-6 spatially
lapply(head(colnames(pcs), 6), 
    \(.) plotCoords(spe, annotate=.) +
    scale_color_gradientn(., colors=pals::jet())) |>
    wrap_plots(nrow=2) & theme(
        plot.title=element_blank(),
        legend.key.width=unit(0.5, "lines"),
        legend.key.height=unit(1, "lines"))

Quality control metrics tend to be low for specific clusters. Their patch-like pattern, in turn, explains the clustering of low-quality spots seen earlier.

In [ ]:
#| code-fold: true
lapply(c("detected", "log_sum", "mt_prop"), \(.)
    plotColData(spe, x=., y="Leiden", color_by="discard", point_size=0.1) +
    scale_x_discrete(limits=names(sort(by(spe[[.]], spe$Leiden, median))))) |>
    wrap_plots(nrow=1, guides="collect") &
    scale_color_manual("discard", values=c("lavender", "purple")) &
    guides(col=guide_legend(override.aes=list(alpha=1, size=3))) &
    theme_minimal() & theme(
        panel.grid.minor=element_blank(), 
        legend.key.size=unit(0, "lines"))

## Signatures

[Note that such gene lists like these may come from many different places - e.g., in-house analyses, other publications matching the research question etc. As such, they may also be read in from a *.csv* file, or stem from upstream analyses. In any case, they should be curated well and interpreted with caution.]{.aside} Rather than investigating single genes, we can also evaluate the expression of sets of genes (e.g., pathway signatures); e.g., malignant tissue may differ in metabolic activity such as glycolysis and fatty acid metabolism, or exhibit increased apoptosis (cell death) etc. Here, we retrieve hallmark gene sets for some biological phenomena from [MSigDB](https://www.gsea-msigdb.org/gsea/msigdb), using the `r BiocStyle::Biocpkg("msigdbr")` package:

In [ ]:
# retrieve hallmark gene sets from 'MSigDB'
db <- msigdbr(species="Homo sapiens", collection="H")
# get list of gene symbols, one element per set
gs <- split(db$ensembl_gene, db$gs_name)
# simplify set identifiers (drop prefix, use lower case)
names(gs) <- tolower(gsub("HALLMARK_", "", names(gs)))
# how many sets?
length(gs) 
# how many genes in each?
range(sapply(gs, length))

[By definition, `AUCell` yields values in \[0,1\]. Larger sets (more genes) are more likely to achieve higher scores by chance (e.g., a gene set of *all* genes would score 1 in any dataset). It is thus unfair to compare them directly. However, spatial distribution, correlation, and relative comparisons between subpopulations etc. are still meaningful.]{.aside} Next, we will score these using `r BiocStyle::Biocpkg("AUCell")` [@Aibar2017-SCENIC], which works in two steps: (i) rank genes for every observation (here, spots), and (ii) compute AUC values for each gene set. In essence, these represent the fraction of genes (within top-ranked genes; default 5%) that are in a given set; i.e., high values correspond to high activity (in terms of coordinated gene expression).

In [ ]:
# realize (sparse) gene expression matrix
mtx <- as(logcounts(spe), "dgCMatrix") 
# use ensembl identifiers as feature names
rownames(mtx) <- rowData(spe)$ID
# build per-spot gene rankings
rnk <- AUCell_buildRankings(mtx, BPPARAM=bp, plotStats=FALSE, verbose=FALSE)
# calculate AUC for each gene set in each spot
auc <- AUCell_calcAUC(geneSets=gs, rankings=rnk, nCores=th, verbose=FALSE)
# add results as spot metadata
colData(spe)[rownames(auc)] <- res <- t(assay(auc)) 

For simplicity, we’ll continue investigating only those 
signatures with the highest score variability across spots:

In [ ]:
var <- colVars(res) # variance across spots
top <- names(tail(sort(var), 8)) # top sets

To summarize, MYC signalling is absent in stromal regions; the fibroblast ring 
surrounding a cancerous patch exhibits EMT, angiogenesis, etc.; INFa response 
and TNFa signalling is patch-like in both stroma and malignant epithelia.

In [ ]:
#| code-fold: true
lapply(top, \(.) {
    spe[[.]] <- scale(spe[[.]]) # scaling
    plotCoords(spe, annotate=.) # plotting
}) |> 
    # arrange & prettify
    wrap_plots(nrow=2, guides="collect") & 
    scale_color_gradientn(
        colors=pals::jet(),
        oob=scales::squish, 
        limits=c(-2.5, 2.5)) & 
    theme(
        legend.key.width=unit(0.5, "lines"), 
        legend.key.height=unit(1, "lines")) 

To ease interpretability, we can stratify `AUCell` scores by spot labels; 
these may stem from an unsupervised or deconvolution-based approach:

In [ ]:
#| layout-ncol: 2
#| code-fold: true
par(mar=c(0,0,0,0))
for (. in c("Leiden", "RCTD")) {
    # aggregate AUC values by cluster
    ks <- spe[[.]]
    pb <- aggregateAcrossCells.se(auc[top, ], ks, assay.type="AUC")
    mu <- sweep(assay(pb, "sums"), 2, pb$counts, `/`)
    colnames(mu) <- levels(ks)
    # visualize as (cluster x set) heatmap
    pheatmap(
        mat=t(mu), scale="column", col=pals::coolwarm(), main=.,
        cellwidth=10, cellheight=10, treeheight_row=5, treeheight_col=5)
}

For the latter, we may instead correlate set scores with proportion estimates 
(rather than discretizing labels according to the dominant subpopulation):

In [ ]:
# correlate 'AUCell' signature scores with subpopulation
# proportion estimates from deconvolution with 'RCTD'
cm <- cor(as.matrix(ws), t(assay(auc[top, ])))

In [ ]:
#| code-fold: true
#| fig-align: center
par(mar=c(0,0,0,0))
pheatmap(cm, 
    col=pals::coolwarm(),
    breaks=seq(-1, 1, length=25),
    cellwidth=10, cellheight=10, 
    treeheight_row=5, treeheight_col=5)

## Appendix

### References {.unnumbered}